---
# Creative Extension — Reward Shaping with SOFA Score

## Motivation

Both Config A and Config B use a **sparse reward**: the agent receives +1 only at patient survival and 0 at every intermediate step. This makes learning difficult — the agent receives almost no feedback during treatment, only at the very end of the episode.

In real ICUs, clinicians don't wait until the patient either survives or dies to evaluate treatment effectiveness. They continuously monitor clinical indicators such as the **SOFA score** (Sequential Organ Failure Assessment), a validated severity score where higher values indicate worse organ function. A decreasing SOFA score during treatment is a meaningful clinical signal that the patient is responding well.

**Reward shaping** adds small intermediate signals based on SOFA score trajectory without changing the optimal policy:

$$r_{\text{shaped}} = r_{\text{original}} + \eta \cdot (\text{SOFA}_{t} - \text{SOFA}_{t+1})$$

If SOFA decreases (patient improving) → small positive bonus.  
If SOFA increases (patient deteriorating) → small negative penalty.  
The terminal survival reward (+1) is unchanged.

## Scope

We apply reward shaping to the **best algorithm in each configuration**:
- **Config A**: Q-Learning (optimised, alpha=0.1) — best model-free tabular agent
- **Config B**: DQN (500k timesteps, lr=1e-4) — best deep RL agent

For each, we train a shaped version and compare against the already-computed original. This requires only **two additional training runs**.

## Key question

Does adding intermediate SOFA-based signals help the agent learn faster and reach a better policy? Does shaping help more in tabular RL (Config A) or deep RL (Config B)?


---
## 15.1 Config A — Shaped Q-Learning

In Config A the SOFA score is not directly in `env.info` but the environment exposes `env.unwrapped._sofa_scores` — a mapping from state index to SOFA score derived from the original MIMIC-III data. We use this to compute the SOFA delta at each step.

We train Q-Learning with the shaped reward for **200k episodes** (reduced from 500k for efficiency — we showed in Section 5 that performance largely plateaus after 200k with the right alpha). The best alpha from the grid search (0.1) is reused.


In [ ]:
# ── Config A: extract SOFA scores from environment ───────────────────────────

env_sofa = make_sepsis_env()
raw_sofa  = env_sofa.unwrapped

# Try to get SOFA scores — fall back to zeros if not available
try:
    sofa_scores = raw_sofa._sofa_scores   # np.ndarray shape (N_STATES,)
    print(f'SOFA scores loaded: shape {sofa_scores.shape}')
    print(f'SOFA range: [{sofa_scores.min():.1f}, {sofa_scores.max():.1f}]')
    print(f'Mean SOFA: {sofa_scores.mean():.2f}')
    SOFA_AVAILABLE = True
except AttributeError:
    # SOFA not directly accessible — use state index as proxy
    # Higher state index loosely correlates with higher severity in this MDP
    sofa_scores = np.linspace(0, 24, N_STATES)
    print('SOFA scores not directly accessible — using state index proxy')
    SOFA_AVAILABLE = False

env_sofa.close()


In [ ]:
# ── Shaped Q-Learning: Config A ──────────────────────────────────────────────

def q_learning_shaped(
    sofa_scores,
    eta=0.05,           # shaping weight — small enough not to dominate survival reward
    n_episodes=200_000, # reduced from 500k for efficiency
    alpha=0.1,          # best alpha from grid search
    gamma=1.0,
    epsilon_start=1.0,
    epsilon_min=0.01,
    seed=SEED,
):
    """
    Q-Learning with SOFA-based reward shaping.

    At each step the reward is augmented:
        r_shaped = r_original + eta * (sofa[s] - sofa[s'])
    A decrease in SOFA (improvement) gives a positive bonus.
    An increase in SOFA (deterioration) gives a negative penalty.

    Parameters
    ----------
    sofa_scores : np.ndarray (N_STATES,) — SOFA score per state
    eta         : float — shaping weight
    n_episodes  : int   — training episodes
    alpha       : float — learning rate (best from grid search)
    gamma       : float — discount factor
    epsilon_start, epsilon_min : float — epsilon schedule
    seed        : int   — random seed

    Returns
    -------
    Q           : np.ndarray (N_STATES, N_ACTIONS)
    returns_log : list of float — episode returns (original reward only)
    sofa_deltas : list of float — mean SOFA delta per episode
    """
    np.random.seed(seed)
    env_train = make_sepsis_env()
    Q = np.zeros((N_STATES, N_ACTIONS))
    returns_log  = []
    sofa_deltas  = []
    epsilon      = epsilon_start
    decay        = (epsilon_start - epsilon_min) / n_episodes

    for ep in tqdm(range(n_episodes), desc='Q-Learning Shaped', leave=False):
        obs, _ = env_train.reset(seed=np.random.randint(100_000))
        s = int(obs)
        total_r, ep_sofa_delta, steps, done = 0.0, 0.0, 0, False

        while not done:
            if np.random.random() < epsilon:
                a = env_train.action_space.sample()
            else:
                a = int(np.argmax(Q[s]))

            obs_next, r, te, tr, _ = env_train.step(a)
            s_next = int(obs_next)
            done   = te or tr

            # Compute SOFA delta — positive if SOFA decreases (improvement)
            sofa_delta = sofa_scores[s] - sofa_scores[s_next]
            ep_sofa_delta += sofa_delta

            # Shaped reward
            r_shaped = r + eta * sofa_delta

            # Q-Learning update with shaped reward
            td_target = r_shaped + gamma * np.max(Q[s_next]) * (not done)
            Q[s, a]  += alpha * (td_target - Q[s, a])

            s       = s_next
            total_r += r      # log original reward only for fair comparison
            steps   += 1

        returns_log.append(total_r)
        sofa_deltas.append(ep_sofa_delta / max(steps, 1))
        epsilon = max(epsilon_min, epsilon - decay)

    env_train.close()
    return Q, returns_log, sofa_deltas


# Run shaped Q-Learning
ETA_A = 0.05
print(f'Training Shaped Q-Learning | eta={ETA_A} | 200k episodes...')
ql_shaped_Q, ql_shaped_returns, ql_shaped_sofa = q_learning_shaped(
    sofa_scores=sofa_scores,
    eta=ETA_A,
    n_episodes=200_000,
    alpha=0.1,
    gamma=GAMMA,
    seed=SEED,
)
ql_shaped_policy  = np.argmax(ql_shaped_Q, axis=1)
ql_shaped_results = evaluate_policy_tabular(ql_shaped_policy)

print(f'Shaped Q-Learning — Survival: {ql_shaped_results["survival_rate"]:.1f}%')
print(f'Original Q-Learning (200k)  — Survival: '
      f'{alpha_grid_results[("Q-Learning", 0.1)]["survival_rate"]:.1f}%')
print(f'Optimised Q-Learning (500k) — Survival: {ql_opt_results["survival_rate"]:.1f}%')


In [ ]:
# ── Plot Config A: shaped vs original Q-Learning ─────────────────────────────

window = 500

# Original Q-Learning at 200k with alpha=0.1 (from grid search — already computed)
ql_orig_200k_returns = alpha_grid_returns[('Q-Learning', 0.1)]
ql_orig_200k_roll    = pd.Series(ql_orig_200k_returns).rolling(window, min_periods=1).mean()
ql_shaped_roll       = pd.Series(ql_shaped_returns).rolling(window, min_periods=1).mean()
ql_sofa_roll         = pd.Series(ql_shaped_sofa).rolling(window, min_periods=1).mean()

fig, axes = plt.subplots(1, 3, figsize=(20, 4))

# Learning curves: shaped vs original
axes[0].plot(ql_orig_200k_roll, color='darkorange', linewidth=2.0,
             label='Q-Learning original')
axes[0].plot(ql_shaped_roll, color='green', linewidth=2.0,
             label=f'Q-Learning shaped (η={ETA_A})')
axes[0].axhline(pi_results['survival_rate'], color='tomato', linestyle='--',
                linewidth=1.5, label=f'PI ceiling ({pi_results["survival_rate"]:.1f}%)')
axes[0].axhline(survival_rate, color='gray', linestyle=':',
                linewidth=1.5, label=f'Random ({survival_rate:.1f}%)')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Rolling mean return')
axes[0].set_title('Config A: Shaped vs Original Q-Learning\nLearning Curves (200k episodes)',
                  fontweight='bold')
axes[0].legend(fontsize=8)

# Survival rate bar comparison
bar_labels = ['Random', 'Q-Lrn\nOriginal\n(200k)', 'Q-Lrn\nShaped\n(200k)',
              'Q-Lrn\nOptimised\n(500k)', 'PI\nCeiling']
bar_values = [survival_rate,
              alpha_grid_results[('Q-Learning', 0.1)]['survival_rate'],
              ql_shaped_results['survival_rate'],
              ql_opt_results['survival_rate'],
              pi_results['survival_rate']]
bar_colors = ['gray', '#fad7a0', 'green', 'darkorange', 'tomato']

bars = axes[1].bar(range(len(bar_labels)), bar_values,
                   color=bar_colors, alpha=0.85, edgecolor='white')
axes[1].set_xticks(range(len(bar_labels)))
axes[1].set_xticklabels(bar_labels, fontsize=8)
axes[1].set_ylabel('Survival Rate (%)')
axes[1].set_title('Config A: Survival Rate Comparison', fontweight='bold')
axes[1].set_ylim(60, 85)
for bar, val in zip(bars, bar_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}%', ha='center', va='bottom',
                 fontsize=8, fontweight='bold')

# SOFA delta during shaped training — shows agent learning to improve patients
axes[2].plot(ql_sofa_roll, color='green', linewidth=1.5, alpha=0.8)
axes[2].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[2].set_xlabel('Episode')
axes[2].set_ylabel('Mean SOFA delta per episode')
axes[2].set_title('Config A: Mean SOFA Delta During Training\n(positive = patient improving)',
                  fontweight='bold')
axes[2].fill_between(range(len(ql_sofa_roll)),
                     ql_sofa_roll, 0,
                     where=ql_sofa_roll > 0,
                     alpha=0.2, color='green', label='Improving')
axes[2].fill_between(range(len(ql_sofa_roll)),
                     ql_sofa_roll, 0,
                     where=ql_sofa_roll < 0,
                     alpha=0.2, color='red', label='Deteriorating')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/creative_configA_shaping.png', bbox_inches='tight')
plt.show()

# Summary
shaping_gain_A = (ql_shaped_results['survival_rate'] -
                  alpha_grid_results[('Q-Learning', 0.1)]['survival_rate'])
print(f'Config A shaping gain: {shaping_gain_A:+.1f}pp '
      f'({alpha_grid_results[("Q-Learning", 0.1)]["survival_rate"]:.1f}% → '
      f'{ql_shaped_results["survival_rate"]:.1f}%)')


---
## 15.2 Config B — Shaped DQN

In Config B the SOFA score is directly available in `info['sofa_score']` at every step — no approximation needed. We wrap the clinical environment to inject the shaped reward and train DQN for **200k timesteps** (reduced from 500k for efficiency — the learning curve showed DQN largely plateaued after 200k anyway).


In [ ]:
# ── Shaped reward wrapper for Config B ───────────────────────────────────────

import gymnasium as gym

class SOFAShapedRewardWrapper(gym.Wrapper):
    """
    Wrapper that augments the step reward with a SOFA-based shaping signal.

    At each step:
        r_shaped = r_original + eta * (sofa_t - sofa_{t+1})

    A decrease in SOFA (patient improving) gives a positive bonus.
    An increase in SOFA (patient deteriorating) gives a negative penalty.
    The terminal survival reward is unchanged.

    Parameters
    ----------
    env : gymnasium.Env — the wrapped clinical environment
    eta : float — shaping weight (default 0.05)
    """
    def __init__(self, env, eta=0.05):
        super().__init__(env)
        self.eta       = eta
        self._prev_sofa = None

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._prev_sofa = info.get('sofa_score', 0.0)
        return obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        curr_sofa = info.get('sofa_score', self._prev_sofa)

        # Shaping: positive if SOFA decreased (improvement)
        sofa_delta  = self._prev_sofa - curr_sofa
        reward_shaped = reward + self.eta * sofa_delta

        self._prev_sofa = curr_sofa
        return obs, reward_shaped, terminated, truncated, info


def make_shaped_clinical_env(eta=0.05):
    """Create the clinical environment with SOFA reward shaping."""
    base_env = make_clinical_env()
    return SOFAShapedRewardWrapper(base_env, eta=eta)


# Test the wrapper
ETA_B = 0.05
test_env = make_shaped_clinical_env(eta=ETA_B)
obs, info = test_env.reset(seed=SEED)
print(f'Shaped clinical env ready.')
print(f'Observation space : {test_env.observation_space}')
print(f'SOFA at reset     : {info.get("sofa_score", "N/A")}')
obs, r, te, tr, info = test_env.step(0)
print(f'Shaped reward after step 0: {r:.4f}')
test_env.close()


In [ ]:
# ── Shaped DQN training: Config B ────────────────────────────────────────────
# 200k timesteps — efficient, DQN largely plateaued by then anyway

N_TIMESTEPS_SHAPED = 200_000

np.random.seed(SEED)
env_dqn_shaped_train = make_shaped_clinical_env(eta=ETA_B)
env_dqn_shaped_eval  = make_clinical_env()   # eval on ORIGINAL env (no shaping)

dqn_shaped_callback = ReturnLoggerCallback(
    eval_env=env_dqn_shaped_eval,
    eval_freq=2000,
    n_eval_episodes=200,
    verbose=1,
)

dqn_shaped_model = DQN(
    policy                 = 'MlpPolicy',
    env                    = env_dqn_shaped_train,
    learning_rate          = 1e-4,
    buffer_size            = 50_000,
    learning_starts        = 1_000,
    batch_size             = 64,
    gamma                  = 1.0,
    target_update_interval = 500,
    exploration_fraction   = 0.3,
    exploration_final_eps  = 0.05,
    verbose                = 0,
    seed                   = SEED,
)

print(f'Training Shaped DQN | eta={ETA_B} | {N_TIMESTEPS_SHAPED:,} timesteps...')
dqn_shaped_model.learn(
    total_timesteps=N_TIMESTEPS_SHAPED,
    callback=dqn_shaped_callback,
)
print('Shaped DQN training complete.')

env_dqn_shaped_train.close()
env_dqn_shaped_eval.close()


In [ ]:
# ── Evaluate shaped DQN on original (unshapped) environment ──────────────────
# Important: always evaluate on the ORIGINAL environment so comparison is fair
# Shaped training, but survival rate measured without the bonus signal

dqn_shaped_results = evaluate_deep_agent(dqn_shaped_model, n_episodes=1000)
print('Shaped DQN — Evaluation on original environment (1000 episodes):')
print(f'  Survival rate : {dqn_shaped_results["survival_rate"]:.1f}%')
print(f'  Mean return   : {dqn_shaped_results["mean_return"]:.4f}')
print(f'  Std return    : {dqn_shaped_results["std_return"]:.4f}')
print()

# Compare to DQN at same timesteps (200k from sensitivity check)
# We use the lr sensitivity 200k run as the baseline for fair comparison
dqn_200k_survival = dqn_results['survival_rate']  # 500k, use as upper bound reference
print(f'Reference — DQN original (500k): {dqn_results["survival_rate"]:.1f}%')
print(f'Shaped DQN (200k)              : {dqn_shaped_results["survival_rate"]:.1f}%')


In [ ]:
# ── Plot Config B: shaped vs original DQN ────────────────────────────────────

dqn_shaped_steps    = [x[0] for x in dqn_shaped_callback.return_log]
dqn_shaped_surv_log = [x[2] for x in dqn_shaped_callback.return_log]
dqn_shaped_ret_log  = [x[1] for x in dqn_shaped_callback.return_log]

window = 20
dqn_shaped_surv_smooth = pd.Series(dqn_shaped_surv_log).rolling(window, min_periods=1).mean()
dqn_shaped_ret_smooth  = pd.Series(dqn_shaped_ret_log).rolling(window, min_periods=1).mean()

# Original DQN at same x-axis length for fair visual comparison
dqn_orig_steps_200k = [x[0] for x in dqn_callback.return_log
                        if x[0] <= N_TIMESTEPS_SHAPED]
dqn_orig_surv_200k  = [x[2] for x in dqn_callback.return_log
                        if x[0] <= N_TIMESTEPS_SHAPED]
dqn_orig_surv_smooth_200k = pd.Series(dqn_orig_surv_200k).rolling(
    window, min_periods=1).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Survival rate: shaped vs original (first 200k steps)
axes[0].plot(dqn_orig_steps_200k, dqn_orig_surv_smooth_200k,
             color='steelblue', linewidth=2.0, label='DQN original')
axes[0].plot(dqn_shaped_steps, dqn_shaped_surv_smooth,
             color='green', linewidth=2.0,
             label=f'DQN shaped (η={ETA_B})')
axes[0].plot(dqn_orig_steps_200k, dqn_orig_surv_200k,
             color='steelblue', linewidth=0.6, alpha=0.2)
axes[0].plot(dqn_shaped_steps, dqn_shaped_surv_log,
             color='green', linewidth=0.6, alpha=0.2)
axes[0].axhline(rand_survival_B, color='gray', linestyle=':',
                linewidth=1.5,
                label=f'Random baseline ({rand_survival_B:.1f}%)')
axes[0].set_xlabel('Training timesteps')
axes[0].set_ylabel('Survival Rate (%)')
axes[0].set_title('Config B: Shaped vs Original DQN\nSurvival Rate (first 200k steps)',
                  fontweight='bold')
axes[0].legend(fontsize=8)

# Bar chart comparison
bar_labels_B = ['Random', 'DQN\nOriginal\n(500k)', 'DQN\nShaped\n(200k)']
bar_values_B = [rand_survival_B,
                dqn_results['survival_rate'],
                dqn_shaped_results['survival_rate']]
bar_colors_B = ['gray', 'steelblue', 'green']

bars = axes[1].bar(range(len(bar_labels_B)), bar_values_B,
                   color=bar_colors_B, alpha=0.85, edgecolor='white')
axes[1].set_xticks(range(len(bar_labels_B)))
axes[1].set_xticklabels(bar_labels_B, fontsize=9)
axes[1].set_ylabel('Survival Rate (%)')
axes[1].set_title('Config B: Survival Rate Comparison', fontweight='bold')
axes[1].set_ylim(55, 80)
for bar, val in zip(bars, bar_values_B):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}%', ha='center', va='bottom',
                 fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/creative_configB_shaping.png', bbox_inches='tight')
plt.show()

shaping_gain_B = dqn_shaped_results['survival_rate'] - rand_survival_B
print(f'Config B shaped DQN vs random: {shaping_gain_B:+.1f}pp')


---
## 15.3 Cross-Config Comparison — Does Shaping Help More in Tabular or Deep RL?

We now compare the effect of reward shaping across both configurations. The key question: does adding intermediate SOFA signals help tabular Q-Learning and deep DQN equally, or does one benefit more than the other?


In [ ]:
# ── Cross-config shaping comparison ──────────────────────────────────────────

shaping_gain_A = (ql_shaped_results['survival_rate'] -
                  alpha_grid_results[('Q-Learning', 0.1)]['survival_rate'])
shaping_gain_B = (dqn_shaped_results['survival_rate'] -
                  dqn_results['survival_rate'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: shaping gain comparison
configs      = ['Config A
(Q-Learning)', 'Config B
(DQN)']
gains        = [shaping_gain_A, shaping_gain_B]
gain_colors  = ['green' if g > 0 else 'tomato' for g in gains]

bars = axes[0].bar(configs, gains, color=gain_colors, alpha=0.85,
                   edgecolor='white', width=0.4)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_ylabel('Survival Rate Change (pp)')
axes[0].set_title('Effect of Reward Shaping\n(positive = shaping helped)',
                  fontweight='bold')
for bar, val in zip(bars, gains):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + (0.1 if val >= 0 else -0.3),
                 f'{val:+.1f}pp', ha='center', va='bottom',
                 fontsize=11, fontweight='bold')

# Right: full survival rate table as grouped bars
categories   = ['Random
Baseline', 'Original
Algorithm', 'Shaped
Algorithm']
configA_vals = [survival_rate,
                alpha_grid_results[('Q-Learning', 0.1)]['survival_rate'],
                ql_shaped_results['survival_rate']]
configB_vals = [rand_survival_B,
                dqn_results['survival_rate'],
                dqn_shaped_results['survival_rate']]

x     = np.arange(len(categories))
width = 0.35

axes[1].bar(x - width/2, configA_vals, width,
            label='Config A (Q-Learning)', color='darkorange',
            alpha=0.85, edgecolor='white')
axes[1].bar(x + width/2, configB_vals, width,
            label='Config B (DQN)', color='steelblue',
            alpha=0.85, edgecolor='white')

# Mark shaped bars with green outline
for idx in [2]:
    axes[1].bar(idx - width/2, configA_vals[idx], width,
                color='none', edgecolor='green', linewidth=2.5)
    axes[1].bar(idx + width/2, configB_vals[idx], width,
                color='none', edgecolor='green', linewidth=2.5)

axes[1].set_xticks(x)
axes[1].set_xticklabels(categories)
axes[1].set_ylabel('Survival Rate (%)')
axes[1].set_title('Survival Rates: Original vs Shaped\n(green outline = shaped version)',
                  fontweight='bold')
axes[1].set_ylim(55, 85)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/creative_shaping_comparison.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Final creative extension summary ─────────────────────────────────────────

print('=' * 70)
print('CREATIVE EXTENSION — REWARD SHAPING SUMMARY')
print('=' * 70)
print()
print(f'Shaping weight (eta): {ETA_A} (Config A)  |  {ETA_B} (Config B)')
print()
print(f'{"":30} {"Original":>12} {"Shaped":>12} {"Change":>10}')
print('-' * 70)
print(f'{"Config A — Q-Learning (200k)":<30} '
      f'{alpha_grid_results[("Q-Learning", 0.1)]["survival_rate"]:>11.1f}% '
      f'{ql_shaped_results["survival_rate"]:>11.1f}% '
      f'{shaping_gain_A:>+9.1f}pp')
print(f'{"Config B — DQN (200k vs 500k)":<30} '
      f'{dqn_results["survival_rate"]:>11.1f}% '
      f'{dqn_shaped_results["survival_rate"]:>11.1f}% '
      f'{shaping_gain_B:>+9.1f}pp')
print('=' * 70)
print()

# Auto-interpretation
if shaping_gain_A > 0 and shaping_gain_B > 0:
    print('Reward shaping improved performance in BOTH configurations.')
    print('SOFA-based intermediate signals help agents learn more efficiently')
    print('in both tabular and deep RL settings.')
    if abs(shaping_gain_A) > abs(shaping_gain_B):
        print(f'The benefit is larger in Config A (+{shaping_gain_A:.1f}pp vs '
              f'+{shaping_gain_B:.1f}pp), suggesting tabular methods benefit')
        print('more from intermediate signals because each state update is')
        print('independent — shaping directly guides state-level Q-values.')
    else:
        print(f'The benefit is larger in Config B (+{shaping_gain_B:.1f}pp vs '
              f'+{shaping_gain_A:.1f}pp), suggesting neural network function')
        print('approximation benefits more from richer gradient signals.')
elif shaping_gain_A > 0 and shaping_gain_B <= 0:
    print('Reward shaping helped Config A (tabular) but not Config B (deep RL).')
    print('This suggests the neural network in DQN already generalises well')
    print('enough from the sparse reward that additional shaping adds noise')
    print('rather than useful signal — particularly given the noisy wrappers')
    print('in Config B that may corrupt the SOFA-based shaping signal itself.')
elif shaping_gain_A <= 0 and shaping_gain_B > 0:
    print('Reward shaping helped Config B (deep RL) but not Config A (tabular).')
    print('DQN benefits from richer gradient signals through its neural network,')
    print('while the tabular Q-table may be sensitive to the additional reward')
    print('noise introduced by the shaping term.')
else:
    print('Reward shaping did not improve either configuration.')
    print('This suggests the sparse terminal reward is already sufficient for')
    print('these algorithms and environments — or that eta needs further tuning.')

print()
print('Clinical interpretation:')
print('An agent trained with SOFA shaping learns to treat patients in a way')
print('that also improves their clinical condition during the episode, not')
print('just maximising the chance of survival at the terminal step.')
print('This aligns with real ICU practice where intermediate organ function')
print('monitoring guides treatment adjustments continuously.')
